In [36]:
!pip install datasets tiktoken transformers python-dotenv


## Part A2: Auditing the Metric
Here we run the intern's old code on the toy dataset and mathematically isolate the distortions caused by the bugs.


In [37]:
import tiktoken
import unicodedata

def analyze_v0(lines, encode, do_lower=True, do_macro=True):
    per_line_fertility = []
    total_tokens, total_words = 0, 0
    for line in lines:
        if do_lower: line = line.lower()
        tokens, words = encode(line), line.split(" ")
        per_line_fertility.append(len(tokens) / max(len(words), 1))
        total_tokens += len(tokens)
        total_words += len(words)
    return sum(per_line_fertility) / len(per_line_fertility) if do_macro else total_tokens / total_words

def analyze_conceptual(lines, encode):
    total_tokens = sum(len(encode(line)) for line in lines)
    return total_tokens / len(lines)

enc = tiktoken.get_encoding("gpt2")
encode = enc.encode

with open("../../starter_kit/corpus_sample/eng_sample.txt", "r") as f:
    eng_lines = [l.strip() for l in f if l.strip()]
with open("../../starter_kit/corpus_sample/hin_sample.txt", "r") as f:
    hin_lines = [l.strip() for l in f if l.strip()]

print(f"{'Scenario':<25} | {'English':<10} | {'Hindi':<10} | {'Ratio':<10}")
print("-" * 65)

eng_base = analyze_v0(eng_lines, encode, do_lower=True, do_macro=True)
hin_base = analyze_v0(hin_lines, encode, do_lower=True, do_macro=True)
print(f"{'Baseline (Intern v0)':<25} | {eng_base:<10.2f} | {hin_base:<10.2f} | {hin_base/eng_base:.2f}x")

eng_lower_fixed = analyze_v0(eng_lines, encode, do_lower=False, do_macro=True)
hin_lower_fixed = analyze_v0(hin_lines, encode, do_lower=False, do_macro=True)
print(f"{'Lowercasing Fixed':<25} | {eng_lower_fixed:<10.2f} | {hin_lower_fixed:<10.2f} | {hin_lower_fixed/eng_lower_fixed:.2f}x")

eng_macro_fixed = analyze_v0(eng_lines, encode, do_lower=True, do_macro=False)
hin_macro_fixed = analyze_v0(hin_lines, encode, do_lower=True, do_macro=False)
print(f"{'Macro-avg Fixed':<25} | {eng_macro_fixed:<10.2f} | {hin_macro_fixed:<10.2f} | {hin_macro_fixed/eng_macro_fixed:.2f}x")

eng_sent = analyze_conceptual(eng_lines, encode)
hin_sent = analyze_conceptual(hin_lines, encode)
print(f"{'Tokens/Sentence Fixed':<25} | {eng_sent:<10.2f} | {hin_sent:<10.2f} | {hin_sent/eng_sent:.2f}x")


Scenario                  | English    | Hindi      | Ratio     
-----------------------------------------------------------------
Baseline (Intern v0)      | 1.27       | 7.45       | 5.89x
Lowercasing Fixed         | 1.23       | 7.45       | 6.06x
Macro-avg Fixed           | 1.25       | 7.40       | 5.91x
Tokens/Sentence Fixed     | 9.60       | 45.90      | 4.78x


## Part A3: Corrected Analysis
Now we run the corrected metric across the entire ~14,000 sentence corpus using `gpt2`, `gpt-4o`, and `xlm-roberta-base`.


In [38]:
import os
import tiktoken
from transformers import AutoTokenizer
from dotenv import load_dotenv

load_dotenv("../.env")

datasets = ["flores", "in22_gen", "in22_conv"]
langs = ["eng", "hin", "tam", "kan"]
corpus_data = {ds: {lang: [] for lang in langs} for ds in datasets}

for ds in datasets:
    for lang in langs:
        path = os.path.join("corpus", ds, f"{lang}.txt")
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                corpus_data[ds][lang] = [l.strip() for l in f if l.strip()]

def evaluate_tokenizer(corpus_data, encode_fn, tok_name):
    print(f"{'Dataset':<10} | {'Tokenizer':<15} | {'Lang':<5} | {'Toks/Sent':<10} | {'Fertility':<10}")
    print("-" * 65)
    
    for ds_name, langs_dict in corpus_data.items():
        if not langs_dict.get("eng"): continue
        
        total_eng_tokens = sum(len(encode_fn(line)) for line in langs_dict["eng"])
        eng_toks_per_sent = total_eng_tokens / len(langs_dict["eng"])
        
        for lang in langs:
            lines = langs_dict.get(lang, [])
            if not lines: continue
            
            toks_per_sent = sum(len(encode_fn(line)) for line in lines) / len(lines)
            fertility = toks_per_sent / eng_toks_per_sent
            
            print(f"{ds_name:<10} | {tok_name:<15} | {lang:<5} | {toks_per_sent:<10.2f} | {fertility:.2f}x")
    print("-" * 65)


### GPT-2 Baseline


In [39]:
enc_gpt2 = tiktoken.get_encoding("gpt2")
evaluate_tokenizer(corpus_data, enc_gpt2.encode, "gpt2")


Dataset    | Tokenizer       | Lang  | Toks/Sent  | Fertility 
-----------------------------------------------------------------
flores     | gpt2            | eng   | 26.72      | 1.00x
flores     | gpt2            | hin   | 198.09     | 7.41x
flores     | gpt2            | tam   | 415.19     | 15.54x
flores     | gpt2            | kan   | 363.05     | 13.59x
in22_gen   | gpt2            | eng   | 32.98      | 1.00x
in22_gen   | gpt2            | hin   | 239.71     | 7.27x
in22_gen   | gpt2            | tam   | 499.88     | 15.16x
in22_gen   | gpt2            | kan   | 449.55     | 13.63x
in22_conv  | gpt2            | eng   | 12.28      | 1.00x
in22_conv  | gpt2            | hin   | 81.10      | 6.60x
in22_conv  | gpt2            | tam   | 172.03     | 14.01x
in22_conv  | gpt2            | kan   | 149.59     | 12.18x
-----------------------------------------------------------------


### GPT-4o (o200k_base) Modern Tokenizer


In [40]:
enc_gpt4o = tiktoken.get_encoding("o200k_base")
evaluate_tokenizer(corpus_data, enc_gpt4o.encode, "gpt-4o")


Dataset    | Tokenizer       | Lang  | Toks/Sent  | Fertility 
-----------------------------------------------------------------
flores     | gpt-4o          | eng   | 26.55      | 1.00x
flores     | gpt-4o          | hin   | 41.77      | 1.57x
flores     | gpt-4o          | tam   | 52.57      | 1.98x
flores     | gpt-4o          | kan   | 52.28      | 1.97x
in22_gen   | gpt-4o          | eng   | 32.77      | 1.00x
in22_gen   | gpt-4o          | hin   | 50.68      | 1.55x
in22_gen   | gpt-4o          | tam   | 66.90      | 2.04x
in22_gen   | gpt-4o          | kan   | 69.36      | 2.12x
in22_conv  | gpt-4o          | eng   | 12.01      | 1.00x
in22_conv  | gpt-4o          | hin   | 17.11      | 1.43x
in22_conv  | gpt-4o          | tam   | 22.74      | 1.89x
in22_conv  | gpt-4o          | kan   | 25.29      | 2.11x
-----------------------------------------------------------------


### XLM-Roberta (Multilingual)


In [41]:
tok_indic = AutoTokenizer.from_pretrained("xlm-roberta-base")
evaluate_tokenizer(corpus_data, lambda s: tok_indic.encode(s, add_special_tokens=False), "xlm-roberta")


Dataset    | Tokenizer       | Lang  | Toks/Sent  | Fertility 
-----------------------------------------------------------------
flores     | xlm-roberta     | eng   | 30.30      | 1.00x
flores     | xlm-roberta     | hin   | 37.77      | 1.25x
flores     | xlm-roberta     | tam   | 40.87      | 1.35x
flores     | xlm-roberta     | kan   | 40.99      | 1.35x
in22_gen   | xlm-roberta     | eng   | 37.12      | 1.00x
in22_gen   | xlm-roberta     | hin   | 45.07      | 1.21x
in22_gen   | xlm-roberta     | tam   | 51.80      | 1.40x
in22_gen   | xlm-roberta     | kan   | 55.06      | 1.48x
in22_conv  | xlm-roberta     | eng   | 13.33      | 1.00x
in22_conv  | xlm-roberta     | hin   | 15.59      | 1.17x
in22_conv  | xlm-roberta     | tam   | 16.32      | 1.22x
in22_conv  | xlm-roberta     | kan   | 19.55      | 1.47x
-----------------------------------------------------------------


### Llama-3 (Modern 128k BPE)


In [42]:
try:
    tok_llama = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", token=os.environ.get("HF_TOKEN"))
    evaluate_tokenizer(corpus_data, lambda s: tok_llama.encode(s, add_special_tokens=False), "llama-3")
except Exception as e:
    print(f"Error loading Llama-3 (Make sure your HF_TOKEN is authorized for the Llama-3 repo!): {e}")


Dataset    | Tokenizer       | Lang  | Toks/Sent  | Fertility 
-----------------------------------------------------------------
flores     | llama-3         | eng   | 26.85      | 1.00x
flores     | llama-3         | hin   | 67.70      | 2.52x
flores     | llama-3         | tam   | 205.31     | 7.65x
flores     | llama-3         | kan   | 237.82     | 8.86x
in22_gen   | llama-3         | eng   | 33.30      | 1.00x
in22_gen   | llama-3         | hin   | 81.58      | 2.45x
in22_gen   | llama-3         | tam   | 246.87     | 7.41x
in22_gen   | llama-3         | kan   | 295.00     | 8.86x
in22_conv  | llama-3         | eng   | 12.27      | 1.00x
in22_conv  | llama-3         | hin   | 27.85      | 2.27x
in22_conv  | llama-3         | tam   | 85.59      | 6.98x
in22_conv  | llama-3         | kan   | 97.81      | 7.97x
-----------------------------------------------------------------


# Recommendation Memo

## Corrected Headline Numbers

After evaluating our Multilingual Eval Corpus (spanning Encyclopedic, News, and Conversational domains) using the corrected semantic metric (**Tokens per Sentence**), we found that tokenizers drastically change the hardware compute requirements for our Indic users.

Below is the complete dataset breakdown of Tokens per Sentence and the cross-language fertility multiplier (relative to English) for our four tokenizers.

### 1. Legacy Baseline (GPT-2)
The old BPE tokenizer heavily penalizes Indic languages, effectively making them 7x to 15x more expensive.
| Dataset | English | Hindi | Tamil | Kannada |
| :--- | :--- | :--- | :--- | :--- |
| **FLORES-200** | 26.72 | 198.09 (7.41x) | 415.19 (15.54x) | 363.05 (13.59x) |
| **IN22-Gen** | 32.98 | 239.71 (7.27x) | 499.88 (15.16x) | 449.55 (13.63x) |
| **IN22-Conv** | 12.28 | 81.10 (6.60x) | 172.03 (14.01x) | 149.59 (12.18x) |

### 2. Modern Standard (GPT-4o)
The massive `o200k_base` vocabulary significantly compresses Indic text, dropping the multiplier to ~1.5x - 2.0x.
| Dataset | English | Hindi | Tamil | Kannada |
| :--- | :--- | :--- | :--- | :--- |
| **FLORES-200** | 26.55 | 41.77 (1.57x) | 52.57 (1.98x) | 52.28 (1.97x) |
| **IN22-Gen** | 32.77 | 50.68 (1.55x) | 66.90 (2.04x) | 69.36 (2.12x) |
| **IN22-Conv** | 12.01 | 17.11 (1.43x) | 22.74 (1.89x) | 25.29 (2.11x) |

### 3. Open Source Standard (Llama-3 128k)
While much better than GPT-2, Llama-3 still struggles with Dravidian languages compared to GPT-4o, costing ~7x - 8x more than English.
| Dataset | English | Hindi | Tamil | Kannada |
| :--- | :--- | :--- | :--- | :--- |
| **FLORES-200** | 26.85 | 67.70 (2.52x) | 205.31 (7.65x) | 237.82 (8.86x) |
| **IN22-Gen** | 33.30 | 81.58 (2.45x) | 246.87 (7.41x) | 295.00 (8.86x) |
| **IN22-Conv** | 12.27 | 27.85 (2.27x) | 85.59 (6.98x) | 97.81 (7.97x) |

### 4. Multilingual Specialist (XLM-Roberta)
The absolute best compression for Indic languages, dropping the penalty to near parity (~1.3x).
| Dataset | English | Hindi | Tamil | Kannada |
| :--- | :--- | :--- | :--- | :--- |
| **FLORES-200** | 30.30 | 37.77 (1.25x) | 40.87 (1.35x) | 40.99 (1.35x) |
| **IN22-Gen** | 37.12 | 45.07 (1.21x) | 51.80 (1.40x) | 55.06 (1.48x) |
| **IN22-Conv** | 13.33 | 15.59 (1.17x) | 16.32 (1.22x) | 19.55 (1.47x) |

*Note: The mathematically correct denominator is "Tokens per Sentence", because a parallel sentence holds the semantic payload of information constant across languages.*

